In [1]:
!pip install -q langgraph langchain-openai

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
    model="google/gemini-2.5-flash",
    max_tokens=500,
    temperature=0.3
)

In [8]:
response = llm.invoke(
    "Say hello in one sentence."
)

print(response.content)

Hello!


In [9]:
from typing import TypedDict

class InvoiceState(TypedDict):
    invoice_data: dict
    validation_result: str
    risk_result: str
    risk_score: int
    ai_analysis: str
    final_report: str

In [10]:
def validate_invoice(state):

    invoice = state["invoice_data"]

    required_fields = [
        "invoice_number",
        "vendor",
        "amount",
        "date"
    ]

    missing = []

    for field in required_fields:
        if field not in invoice:
            missing.append(field)

    if missing:
        state["validation_result"] = (
            f"Missing fields: {', '.join(missing)}"
        )
    else:
        state["validation_result"] = (
            "Invoice validation successful"
        )

    return state

In [11]:
def risk_analysis(state):

    amount = float(
        state["invoice_data"].get("amount", 0)
    )

    risks = []
    risk_score = 0

    if amount > 100000:
        risks.append("High Value Invoice")
        risk_score += 8

    if amount <= 0:
        risks.append("Invalid Amount")
        risk_score += 10

    if amount > 0 and amount <= 100000:
        risk_score += 2

    if len(risks) == 0:
        state["risk_result"] = (
            "No major risks detected"
        )
    else:
        state["risk_result"] = (
            ", ".join(risks)
        )

    state["risk_score"] = risk_score

    return state

In [12]:
def ai_analysis(state):

    prompt = f"""
You are a financial invoice auditor.

Invoice:
{state['invoice_data']}

Validation:
{state['validation_result']}

Risk Analysis:
{state['risk_result']}

Risk Score:
{state['risk_score']}/10

Provide:

1. Summary
2. Potential Concerns
3. Recommendations
"""

    response = llm.invoke(prompt)

    state["ai_analysis"] = response.content

    return state

In [17]:
def generate_report(state):

    report = f"""
AI INVOICE ANALYSIS REPORT

Invoice:
{state['invoice_data']}

Validation:
{state['validation_result']}

Risk Analysis:
{state['risk_result']}

Risk Score:
{state['risk_score']}/10

AI Insights:
{state['ai_analysis']}
"""

    state["final_report"] = report

    return state

In [18]:
from langgraph.graph import StateGraph, END

builder = StateGraph(InvoiceState)

builder.add_node("validation", validate_invoice)
builder.add_node("risk", risk_analysis)
builder.add_node("analysis", ai_analysis)
builder.add_node("report", generate_report)

builder.set_entry_point("validation")

builder.add_edge("validation", "risk")
builder.add_edge("risk", "analysis")
builder.add_edge("analysis", "report")
builder.add_edge("report", END)

graph = builder.compile()

print("LangGraph Created Successfully")

LangGraph Created Successfully


In [20]:
invoice = {
    "invoice_number": "INV-001",
    "vendor": " Fintom8 AI ",
    "amount": 150000,
    "date": "2026-06-13"
}

result = graph.invoke({
    "invoice_data": invoice
})

print(result["final_report"])


AI INVOICE ANALYSIS REPORT

Invoice:
{'invoice_number': 'INV-001', 'vendor': ' Fintom8 AI ', 'amount': 150000, 'date': '2026-06-13'}

Validation:
Invoice validation successful

Risk Analysis:
High Value Invoice

Risk Score:
8/10

AI Insights:
Here's the audit report for the provided invoice:

---

**Invoice Audit Report**

**1. Summary**

The invoice, INV-001 from Fintom8 AI, dated 2026-06-13, has been successfully validated. The invoice amount is 150,000. The primary risk identified is its high value.

**2. Potential Concerns**

*   **High Value Transaction:** An amount of 150,000 is significant and typically warrants increased scrutiny to prevent financial loss due to errors, fraud, or overpayment.
*   **Vendor Name Discrepancy/Typo (Minor):** The vendor name "Fintom8 AI" contains a '8' instead of a 'B' or 'AI' might be a typo or an unusual naming convention. While not a direct risk, it could indicate a data entry error or a less formal vendor name, which might be worth verifying ag